# 03: Why the transfer works or fails (Test 6)

The classifier never sees waveforms, only the 11 RF-v2 feature values. So the question
for every feature is: do DAS events land in the range the model saw on seismometers?

- Inside the seismometer range: the model has seen values like this before.
- Outside it: the model is guessing, and that feature is the first suspect when a DAS
  event gets the wrong label.

**Pass criterion (Test 6):** every gap between seismometer and DAS feature values is
either explained or listed as a limitation. Check date: Nov 25.

**Inputs** (feature CSVs, one row per window, one column per feature, plus `label`):

| What | Who | Where this notebook looks |
|---|---|---|
| Seismometer features, GTUNE (UNE) + CREW (EQ), with `dist_deg` | Ahnaf | `data/features/seismometer/*.csv` |
| 33 stacked Monterey Bay events | Sammy | `data/features/das_stacked/*.csv` |
| Single DAS channels (notebook 02, section 9) | Sammy | `data/features/*_features.csv` |
| Synthetic DAS explosions from GTUNE | Himi | `data/features/synthetic_une/*.csv` |
| Real DAS blasts (DAG-2, Viens et al. 2025) | Sofian + Mahmoud | `data/features/blasts/*.csv` |

Any input that is missing is skipped, so this runs at every stage. Rerun it once each file lands.

In [ ]:
import sys
sys.path.append("..")  # so 'src' imports work from inside notebooks/

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.compare import (FEATURES_11, HANDOUT_MEDIANS, load_features, handout_errors,
                         bin_medians, range_table, plot_overlays)

FEAT = Path("../data/features")
ASSETS = Path("../assets")

seis      = load_features(str(FEAT / "seismometer" / "*.csv"), source="seismometer")
das_stack = load_features(str(FEAT / "das_stacked" / "*.csv"), label="EQ", source="DAS stacked")
das_chan  = load_features(str(FEAT / "*_features.csv"), label="EQ", source="DAS single channel")
synth     = load_features(str(FEAT / "synthetic_une" / "*.csv"), label="UNE", source="synthetic DAS")
blasts    = load_features(str(FEAT / "blasts" / "*.csv"), label="UNE", source="real DAS blasts")

inputs = {"seismometer": seis, "DAS stacked": das_stack, "DAS single channel": das_chan,
          "synthetic DAS UNE": synth, "real DAS blasts": blasts}
for name, df in inputs.items():
    print(f"{name:20s} {'MISSING' if df.empty else f'{len(df):6d} rows'}")

seis_eq = seis[seis["label"] == "EQ"] if not seis.empty else seis
seis_une = seis[seis["label"] == "UNE"] if not seis.empty else seis
das = das_stack if not das_stack.empty else das_chan   # stacked events are the real test unit
print(f"\nDAS rows used below: {das['source'].iloc[0] if not das.empty else 'none'}")

## 1. The two handout errors

Checked on our own feature rows, not just on paper:

- `peak` is the largest absolute sample. After peak normalisation it is 1.0 on every window, so it carries no information.
- `crest_factor` (Appendix B) is Peak / RMS, the same formula as `peak_rms_ratio` (Appendix A).

That is why we train on 11 features, not 13.

In [ ]:
for name, df in inputs.items():
    if not df.empty and "peak" in df:
        print(name, handout_errors(df))

## 2. Our seismometer features vs the handout

The handout lists seismometer feature medians per distance bin. If our GTUNE + CREW features are built the
same way, our medians should land close to his. A big mismatch means our preprocessing differs from his, and
that has to be fixed before any DAS comparison means anything.

In [ ]:
if seis.empty or "dist_deg" not in seis:
    print("Waiting on the seismometer features (with dist_deg) from Ahnaf.")
else:
    display(bin_medians(seis).round(2))

## 3. Where the DAS earthquakes sit

DAS events are earthquakes, so they are compared with the seismometer **earthquakes** (CREW).
`DAS pct` is the percentile of the DAS median within the seismometer values (50 = right in the middle);
`DAS inside` is the share of DAS rows inside the middle 90% of seismometer values.

Until the seismometer features exist, the fallback below compares the DAS median with the handout's
under-5° median (the Monterey Bay events are local).

In [ ]:
if das.empty:
    print("No DAS features yet.")
elif seis_eq.empty:
    rows = [{"feature": f, "DAS median": das[f].median(), "handout <5 median": m["<5"]}
            for f, m in HANDOUT_MEDIANS.items()]
    display(pd.DataFrame(rows).set_index("feature").round(2))
else:
    eq_table = range_table(seis_eq, das)
    display(eq_table.round(2))
    print("Features to explain:", list(eq_table.index[eq_table["verdict"] != "inside"]))

## 4. Figure for the slides and the report

One panel per feature. Dashed lines are the handout's per-bin seismometer medians.

In [ ]:
groups = {"Seismometer EQ": seis_eq, "Seismometer UNE": seis_une, "DAS": das}
if any(not g.empty for g in groups.values()):
    fig = plot_overlays(groups, title="RF-v2 features: seismometer vs DAS",
                        path=str(ASSETS / "fig_feature_compare.png"))
    plt.show()

## 5. The explosion side

Synthetic DAS explosions and the real DAS blasts are compared with the seismometer **explosions** (GTUNE).
If the synthetic and real DAS explosions land in different places, the GTUNE-to-DAS conversion is the suspect.

In [ ]:
for name, df in {"synthetic DAS UNE": synth, "real DAS blasts": blasts}.items():
    if df.empty or seis_une.empty:
        print(f"{name}: waiting on {'this file' if df.empty else 'GTUNE features'}")
    else:
        print(name); display(range_table(seis_une, df, ref_label="UNE").round(2))

## Reading the results

- **inside**: DAS looks like seismometer data for this feature. No explanation needed.
- **edge**: DAS overlaps but is shifted. Say which way and why (e.g. DAS measures strain rate, which weights high frequencies more than velocity).
- **OUTSIDE**: the model has never seen values like this. Explain it, or list it as a limitation.

On single channels we already expect `duration_10`, `duration_20` and `energy_centroid_time` to come out
OUTSIDE, because a single channel is mostly noise (see `fig_features_three.png`). The stacked events are
the real test.